https://milvus.io/docs/es/milvus_lite.md

# Configuración global


# Conexión a Milvus Lite

 COLECCIÓN 1  "conocimiento"

In [ ]:

from pathlib import Path

BASE_DIR = Path.cwd() / "pmai-model-vision-language"
MILVUS_DB_PATH = "./androide_milvus.db"
EMBED_MODEL_NAME  = "paraphrase-multilingual-MiniLM-L12-v2"
EMBED_DIM         = 384
COL_CONOCIMIENTO  = "conocimiento"
COL_INTERACCIONES = "interacciones"
UMBRAL_CONOCIMIENTO = 0.70


In [135]:
from pymilvus import MilvusClient
client = MilvusClient("./androide_milvus.db")

primer schema de la coleccion a


In [136]:
from pymilvus import DataType, MilvusClient
import numpy as np

schema = client.create_schema(
    auto_id=True,                
    enable_dynamic_field=False,  
)


schema = client.create_schema(auto_id=True, enable_dynamic_field=False)
schema.add_field(field_name="id",        datatype=DataType.INT64, is_primary=True)
schema.add_field(field_name="pregunta",  datatype=DataType.VARCHAR, max_length=1000)
schema.add_field(field_name="respuesta", datatype=DataType.VARCHAR, max_length=3000)
schema.add_field(field_name="movimientos",   datatype=DataType.VARCHAR, max_length=500)
schema.add_field(field_name="vector",    datatype=DataType.FLOAT_VECTOR, dim=EMBED_DIM)



{'auto_id': True, 'description': '', 'fields': [{'name': 'id', 'description': '', 'type': <DataType.INT64: 5>, 'is_primary': True, 'auto_id': False}, {'name': 'pregunta', 'description': '', 'type': <DataType.VARCHAR: 21>, 'params': {'max_length': 1000}}, {'name': 'respuesta', 'description': '', 'type': <DataType.VARCHAR: 21>, 'params': {'max_length': 3000}}, {'name': 'movimientos', 'description': '', 'type': <DataType.VARCHAR: 21>, 'params': {'max_length': 500}}, {'name': 'vector', 'description': '', 'type': <DataType.FLOAT_VECTOR: 101>, 'params': {'dim': 384}}], 'enable_dynamic_field': False, 'enable_namespace': False}

In [137]:
client.create_collection(
    collection_name=COL_CONOCIMIENTO,  
    schema=schema,                       
)

In [138]:
index_params = client.prepare_index_params()

index_params.add_index(
    field_name="vector",
    index_type="HNSW",          
    metric_type="COSINE",       
    params={"M": 16, "efConstruction": 200},   
)

client.create_index(
    collection_name=COL_CONOCIMIENTO,
    index_params=index_params,
)



In [139]:
client.load_collection(collection_name=COL_CONOCIMIENTO)

In [140]:
from sentence_transformers import SentenceTransformer
modelo_embed = SentenceTransformer(EMBED_MODEL_NAME)

Loading weights: 100%|█████████████████████████████████| 199/199 [00:00<00:00, 8106.55it/s]


In [ ]:
# este es como el insert que se maneja en una base de datos normalita por si quieren meterle preguntas  especificas :)
# esos si sigan la estructura de  la coleccion  esta en el  diccionario
# nunca usen drop por que eso elimina la coleccion 
# usen delete con su id


#objetos y una base datos
#api de wikipedia para sacar informacion de wikipedia y meterla a la base de datos
#ingesta por apis
#bases de datos entrenadas pregunta respuesta


preguntas = [
    {
        "pregunta": "¿Quien eres?",
        "respuesta": "Soy R-one un androide diseñado por la universidad libre ,para ayudarte en lo que necesites",
        "movimientos": "mueve el brazo derecho"
    },

]

data = []
for item in preguntas:
    vector = modelo_embed.encode(item["pregunta"]).tolist()  
    data.append({
        "pregunta":  item["pregunta"],
        "respuesta": item["respuesta"],
        "vector":    vector,
        "movimientos": item["movimientos"]
    })

client.insert(collection_name=COL_CONOCIMIENTO, data=data)


{'insert_count': 1, 'ids': [1], 'cost': 0}

In [ ]:
# este es el query  para que comprueben lo que le añaden a la base mas adelante por si quieren eliminar algo se puede con el id 
#por favor no borren nada que ustedes no sepan lo que es porque se puede romper la base de datos y no queremos eso  :)
resultados = client.query(
    collection_name=COL_CONOCIMIENTO,
    filter="",                              
    output_fields=["pregunta", "respuesta"], 
    limit=10,
)

for query in resultados:
    print(query)

{'id': 1, 'pregunta': '¿Quien eres?', 'respuesta': 'Soy R-one un androide diseñado por la universidad libre ,para ayudarte en lo que necesites'}


I0607 22:17:49.323410 1110793 chttp2_transport.cc:1369] ipv4:127.0.0.1:60870: Got goaway [11] err=UNAVAILABLE:GOAWAY received; Error code: 11; Debug Text: too_many_pings {grpc_status:14, http2_error:11}
E0607 22:17:49.323492 1110793 chttp2_transport.cc:1401] ipv4:127.0.0.1:60870: Received a GOAWAY with error code ENHANCE_YOUR_CALM and debug data equal to "too_many_pings". Current keepalive time (before throttling): 10000ms


In [143]:
#borre el 5 por que me equivoque al insertar y no quiero eso en la base de datos  :)
#siempre comprobar con el query lo que hagan antes de eliminar algo por que no queremos perder datos importantes  :) 
#y por favor no borren nada que ustedes no sepan lo que es porque se puede romper la base de datos y no queremos eso  :)
client.delete(
    collection_name=COL_CONOCIMIENTO,
    ids=[5],  
)

[5]

### pueden hacer un proceso de reach para probar la base vectorial :)

# ACA va la otra coleccion


In [ ]:
from pymilvus import DataType, MilvusClient
import numpy as np

schema_interacciones = client.create_schema(
    auto_id=True,                
    enable_dynamic_field=False,  
)


schema_interacciones.add_field(field_name="id",datatype=DataType.INT64, is_primary=True,)
schema_interacciones.add_field(field_name="texto_clave",datatype=DataType.VARCHAR, max_length=2000)
schema_interacciones.add_field(field_name="pregunta",datatype=DataType.VARCHAR, max_length=1000)
schema_interacciones.add_field(field_name="scene_summary",datatype=DataType.VARCHAR, max_length=1500)
schema_interacciones.add_field(field_name="respuesta",datatype=DataType.VARCHAR, max_length=3000)
schema_interacciones.add_field(field_name="movimientos",datatype=DataType.VARCHAR, max_length=500)
schema_interacciones.add_field(field_name="vector",datatype=DataType.FLOAT_VECTOR, dim=EMBED_DIM)




client.create_collection(
    collection_name=COL_INTERACCIONES,  
    schema=schema_interacciones,                       
)


index_params = client.prepare_index_params()

index_params.add_index(
    field_name="vector",
    index_type="HNSW",          
    metric_type="COSINE",       
    params={"M": 16, "efConstruction": 200},   
)

client.create_index(
    collection_name=COL_INTERACCIONES,
    index_params=index_params,
)


client.load_collection(collection_name=COL_INTERACCIONES)



In [ ]:
#hacer lo mismo solo adecuar el diccionario a la coleccion adecuada

# estos son los datos
#chema_interacciones.add_field(field_name="id",datatype=DataType.INT64, is_primary=True,)
#schema_interacciones.add_field(field_name="texto_clave",datatype=DataType.VARCHAR, max_length=2000)
#schema_interacciones.add_field(field_name="pregunta",datatype=DataType.VARCHAR, max_length=1000)
#schema_interacciones.add_field(field_name="scene_summary",datatype=DataType.VARCHAR, max_length=1500)
#schema_interacciones.add_field(field_name="respuesta",datatype=DataType.VARCHAR, max_length=3000)
#schema_interacciones.add_field(field_name="movimientos",datatype=DataType.VARCHAR, max_length=500)
#schema_interacciones.add_field(field_name="vector",datatype=DataType.FLOAT_VECTOR, dim=EMBED_DIM)


preguntas = [
    {
        "pregunta": "¿Quien eres?",
        "respuesta": "Soy R-one un androide diseñado por la universidad libre ,para ayudarte en lo que necesites",
        "movimientos": "mueve el brazo derecho"
    },

]

data = []
for item in preguntas:
    vector = modelo_embed.encode(item["pregunta"]).tolist()  
    data.append({
        "pregunta":  item["pregunta"],
        "respuesta": item["respuesta"],
        "vector":    vector,
        "movimientos": item["movimientos"]
    })

client.insert(collection_name=COL_CONOCIMIENTO, data=data)
